In [4]:
pip install transformers datasets torch accelerate evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [9]:
# 匯入必要的庫
from datasets import load_dataset  # 用來載入資料集
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments  # BERT 相關工具
import torch  # PyTorch 框架
import evaluate  # 用來評估模型效能

# Step 1: 載入 IMDb 資料集（情感分析：正面/負面評論）
dataset = load_dataset("imdb")
train_dataset = dataset["train"].shuffle(seed=42).select(range(500))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(100))

# Step 2: 載入 BERT Tokenizer（用來將文字轉換成模型輸入）
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 定義 Tokenization 函數
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)  # 限制長度為 128 tokens

# 應用 Tokenization 到資料集
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Step 3: 載入 BERT 模型（用於序列分類，num_labels=2 表示二元分類）
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)




################################-TO DO-#################################################
# Step 4: 設定訓練參數
training_args = TrainingArguments(
    output_dir="./results",  # 輸出目錄
    num_train_epochs=4,  
    per_device_train_batch_size=14,  # 每個裝置的 batch size
    per_device_eval_batch_size=14,
    warmup_steps=100,  # 學習率 warmup
    weight_decay=0.01,  # 權重衰減
    logging_dir="./logs",  # 日誌目錄
    eval_strategy="epoch",  # 每個 epoch 評估一次
    save_strategy="epoch",  # 每個 epoch 保存模型
    load_best_model_at_end=True,  # 訓練結束載入最佳模型
    report_to="none",
)
#########################################################################################




# Step 5: 定義評估指標（使用 accuracy）
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return metric.compute(predictions=predictions, references=labels)

# Step 6: 使用 Trainer API 進行訓練
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# 開始訓練
trainer.train()

# Step 7: 評估模型
results = trainer.evaluate()
print("評估結果:", results)

# Step 8: 儲存模型（可選）
trainer.save_model("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.735592,0.480000
2,No log,0.606173,0.680000
3,No log,0.759148,0.710000
4,No log,0.439795,0.800000


評估結果: {'eval_loss': 0.43979477882385254, 'eval_accuracy': 0.8, 'eval_runtime': 0.3362, 'eval_samples_per_second': 297.424, 'eval_steps_per_second': 23.794, 'epoch': 4.0}


('./fine_tuned_bert/tokenizer_config.json',
 './fine_tuned_bert/special_tokens_map.json',
 './fine_tuned_bert/vocab.txt',
 './fine_tuned_bert/added_tokens.json')

In [10]:
# Step 9: 使用 fine-tuned 模型預測 CSV 資料
import pandas as pd
from transformers import pipeline

# 載入 fine-tuned 模型
classifier = pipeline("sentiment-analysis", model="./fine_tuned_bert")

# 讀取 CSV 檔案
df = pd.read_csv("data.csv")
predictions = []

for idx, row in df.iterrows():
    text = row['data']
    result = classifier(text)
    predictions.append({
        'text': text,
        'label': result[0]['label'],
        'confidence': result[0]['score']
    })
    
# 將預測結果轉換為 DataFrame
results_df = pd.DataFrame(predictions)
print(results_df)


Device set to use cuda:0


                                                text    label  confidence
0  This movie is sooooo nice!! I should watch it ...  LABEL_1    0.912474
1  Absolutely loved the soundtrack, it fit perfec...  LABEL_1    0.943188
2         One of the best films I’ve seen this year!  LABEL_1    0.944665
3   Amazing visuals, the cinematography is stunning.  LABEL_1    0.955942
4         A masterpiece, truly unforgettable cinema.  LABEL_1    0.958138
5   I wouldn’t recommend it, felt too long and slow.  LABEL_0    0.845415
6        Characters were shallow and underdeveloped.  LABEL_0    0.745563
7         Not worth the hype, kind of disappointing.  LABEL_0    0.831664
8     The story didn’t make sense, left me confused.  LABEL_0    0.628148
9  It was painful to sit through, very disappoint...  LABEL_0    0.776455
